# Preprocessing for Clustering

This notebook creates the first model-ready preprocessing output from the customer-level feature table. It does not run clustering or modelling.

## Imports

Use the reusable preprocessing functions from `src/preprocessing.py`.

In [ ]:
import sys

import pandas as pd

sys.path.append("../src")

from preprocessing import FINAL_MODEL_COLUMNS, clean_feature_values, preprocess_for_clustering, select_model_features


## Load Customer Features

Load the combined customer-level feature table created during feature engineering.

In [ ]:
customer_features = pd.read_csv("../data/processed/customer_features_info.csv")
customer_features.head()


## Initial Checks

Check shape, duplicate customer IDs, and missing values before preprocessing.

In [ ]:
print(f"Rows before preprocessing: {customer_features.shape[0]:,}")
print(f"Columns before preprocessing: {customer_features.shape[1]:,}")
print(f"Duplicated customer_id before preprocessing: {customer_features['customer_id'].duplicated().sum():,}")

In [ ]:
missing_before = customer_features.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

## Clean and Select Features

Clean invalid and missing values, then remove non-model columns while keeping `customer_id` for traceability.

In [ ]:
cleaned_customer_features = clean_feature_values(customer_features)
selected_customer_features = select_model_features(cleaned_customer_features)

removed_columns = [
    column for column in customer_features.columns
    if column not in selected_customer_features.columns
]

removed_columns


In [ ]:
missing_after_cleaning = selected_customer_features.isna().sum().sort_values(ascending=False)
missing_after_cleaning[missing_after_cleaning > 0]

## Preprocess Model Features

Create the final compact model input. Gender and degree fields are excluded from clustering, and the 20 numeric business features are scaled with `RobustScaler`. `customer_id` is preserved only for traceability.


In [ ]:
selected_model_features = preprocess_for_clustering(customer_features)

expected_columns = ["customer_id", *FINAL_MODEL_COLUMNS]
if selected_model_features.columns.tolist() != expected_columns:
    raise ValueError("selected_model_features.csv does not match the final compact feature specification.")

selected_model_features.head()


## Final Validation

Confirm the preprocessed output keeps all customers, has no duplicate IDs, and has no missing values in model features.

In [ ]:
preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "rows before preprocessing",
            "rows after preprocessing",
            "columns before preprocessing",
            "selected columns including customer_id",
            "duplicated customer_id after preprocessing",
            "missing values in selected model features",
        ],
        "value": [
            customer_features.shape[0],
            selected_model_features.shape[0],
            customer_features.shape[1],
            selected_model_features.shape[1],
            selected_model_features["customer_id"].duplicated().sum(),
            selected_model_features.drop(columns=["customer_id"]).isna().sum().sum(),
        ],
    }
)

preprocessing_validation


In [ ]:
missing_after = selected_model_features.drop(columns=["customer_id"]).isna().sum().sort_values(ascending=False)
missing_after[missing_after > 0]


In [ ]:
selected_model_features.columns.to_frame(index=False, name="feature_column")


## Save Preprocessed Feature Table

Save the preprocessed customer-level feature table. This file keeps `customer_id` for traceability, but clustering should use the remaining feature columns.

In [ ]:
selected_model_features.to_csv("../data/processed/selected_model_features.csv", index=False)
print("Saved ../data/processed/selected_model_features.csv")


## Preprocessing Notes

- `customer_id` is kept in the final file for traceability, but it is excluded from model feature scaling.
- Basket-derived features are excluded from clustering and kept for post-clustering profiling and promotions.
- Gender and degree fields are excluded from the final clustering input after feature sensitivity checks.
- Numeric missing values are filled with the median, and categorical missing values are filled with `Unknown` before selection.
- Invalid `percentage_of_products_bought_promotion` values are clipped to the `0` to `100` range.
- The 20 final numeric business features are scaled with `RobustScaler`.
- No clustering or modelling is performed in this notebook.
